In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import factorial

In [ ]:
'''
------------------------------------------
            SETTINGS
------------------------------------------
'''
plt.style.use('fivethirtyeight')
plt.rcParams['font.family'] = 'PT Sans'
plt.rcParams['font.monospace'] = 'Ubuntu Mono'
plt.rcParams['font.size'] = 14
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12
plt.rcParams['legend.fontsize'] = 14
plt.rcParams['figure.titlesize'] = 12

dpi = 100

In [ ]:
def get_filename(filename: str, lecture_id: int = 1, file_extension: str = '.png') -> str:
    return f"L{lecture_id}_{filename}{file_extension}"


def save_figure(outfile, outdir=None, dpi=100):
    """Save the current matplotlib figure if outfile is not None.
    """
    if outfile is None:
        return
    path = f"{outdir}{outfile}"
    plt.savefig(path, dpi=dpi, bbox_inches='tight', pad_inches=0.1)
    print(f"Figure saved in {path}")

In [ ]:
outdir = '../figures/'
lecture_id = 1

In [ ]:
seed = 10
rng = np.random.default_rng(seed)  # single RNG reused (and reseeded) throughout the notebook

# 1. Rolling a dice
Let's simulate a stochastic process of rolling a dice.

In [ ]:
outfile = get_filename('rolling_dice', lecture_id=lecture_id)

In [ ]:
T = 20  # number of draws
y = rng.choice(np.arange(1, 7), size=T, replace=True)
x = np.arange(T)

In [ ]:
plt.scatter(x, y, s=100)
plt.step(x, y, where='mid')

plt.xlabel('Iteration id')
plt.ylabel('Number selected')
plt.xticks(x)
plt.yticks(y)

plt.tight_layout()
save_figure(outfile, outdir=outdir, dpi=dpi)
plt.show()

### 1.1 Increments
Let's take a look at the increments:  
$X_{n+1} - X_{n}$ ,  
where $X_n \in \{1,\dots,6\}$ is the result at time $n$.

In [ ]:
outfile = get_filename('rolling_dice_increments', lecture_id=lecture_id)

In [ ]:
y_increments = np.diff(y)

In [ ]:
plt.scatter(x[1:], y_increments, s=100)
plt.step(x[1:], y_increments, where='mid')
plt.plot(x, np.zeros(len(x)), color='grey', ls='--', zorder=0, lw=2)

plt.xlabel('Iteration id')
plt.ylabel('Increment')
plt.xticks(x)
plt.yticks(y_increments)

plt.tight_layout()
save_figure(outfile, outdir=outdir, dpi=dpi)
plt.show()

In general they can be positive, or negative and fluctuate between a maximum and a minimum value.

### 1.2 Analyzing the empirical distributions
To better understand the process, let's investigate some relevant quantities.  
As we are dealing with random variables, it is useful to observe distributions.

In [ ]:
outfile = get_filename('rolling_dice_dist', lecture_id=lecture_id)

If we want to investigate a probability distribution, we need to consider a large enough number of samples.  
Hence, let's increase the number of draws $T$.

In [ ]:
T = 1000  # number of draws
y = rng.choice(np.arange(1, 7), size=T, replace=True)
y_increments = np.diff(y)
x = np.arange(T)

- What are the most frequent numbers drawn
- How is the increment distributed?

In [ ]:
plt.figure(figsize=(6.5, 3))

plt.subplot(1, 2, 1)
plt.hist(y, bins=6, rwidth=0.5)
plt.ylabel('Frequency')
plt.xlabel('Number selected')
plt.xticks(np.arange(1, 7))

plt.subplot(1, 2, 2)
plt.hist(y_increments, bins=11, rwidth=0.5)
plt.ylabel('Frequency')
plt.xlabel('Increment')
plt.xticks(np.arange(-5, 6))

plt.tight_layout()
save_figure(outfile, outdir=outdir, dpi=dpi)
plt.show()

### 1.3 Turning a dice rolling into a _counting process_
The process was not a counting process.   
For this we need: 
- Positive $N(t) \geq 0$
- Non-decreasing $N(s+t) - N(s) \geq 0$

In [ ]:
outfile = get_filename('rolling_dice_counting', lecture_id=lecture_id)

Let's first generate some random dice draws.

In [ ]:
T = 20  # number of draws
y = rng.choice(np.arange(1, 7), size=T, replace=True)

Now let's make it a counting process!  
How?  
By introducing a $N(t)$ equal to the sum of the numbers drawn.

In [ ]:
N = np.cumsum(y)
x = np.arange(len(N))
n_increments = np.diff(N)

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.scatter(x, N, s=10)
plt.step(x, N, where='mid')
plt.xlabel('Iteration id')
plt.ylabel('N(t)')

plt.subplot(1, 2, 2)
plt.scatter(x[1:], n_increments, s=10)
plt.step(x[1:], n_increments, where='mid')
plt.plot(x, np.zeros(len(x)), color='grey', ls='--', zorder=0, lw=2)
plt.xlabel('Iteration id')
plt.ylabel('Increment N(t)')
plt.xticks(x)
plt.yticks(n_increments)

plt.tight_layout()
save_figure(outfile, outdir=outdir, dpi=dpi)
plt.show()

# 2. Poisson process (PP)

Let's generate events using the definition of a Poisson process.

In [ ]:
seed = 10
rng = np.random.default_rng(seed)

Here we also want to generate more than one sample of the process, so we play also with the variable $M$.

In [ ]:
rate, delta_t = 10, 1
T = 20    # max time window
M = 1000  # number of samples of the process

Now we can draw increments from a Poisson of rate $\lambda \, t$.  
And we repeat this $M$ times.

In [ ]:
y_increments = rng.poisson(rate * delta_t, size=(M, T))
y = np.cumsum(y_increments, axis=1)
x = np.arange(T)

y_increments.shape, y.shape

In [ ]:
outfile = get_filename('pp_basic', lecture_id=lecture_id)

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
selected_samples = rng.choice(np.arange(M), 5)
for idx in selected_samples:
    plt.step(x, y[idx], where='mid', label=f'Sample {idx}', alpha=0.7)

plt.xlabel('Iteration id')
plt.ylabel('N(t)')
plt.xticks(x)
plt.legend()

plt.subplot(1, 2, 2)
idx = selected_samples[-1]
plt.step(x, y_increments[idx], where='mid', label=f'Sample {idx}')
plt.plot(x, np.zeros(len(x)), color='grey', ls='--', zorder=0, lw=2)

plt.xlabel('Iteration id')
plt.ylabel('Increment N(t)')
plt.xticks(x)
plt.legend()

plt.tight_layout()
save_figure(outfile, outdir=outdir, dpi=dpi)
plt.show()

It does not really tell us anything about being _Poisson_.  
Let's generate poisson numbers with parameter $\lambda t$ and compare with the empirical increments $N(s+t)-N(s)$.

In [ ]:
def poisson_pmf(x, lmbda):
    return (np.power(lmbda, x) / factorial(x)) * np.exp(-lmbda)


xs = np.arange(y_increments.max() + 5)
ps = poisson_pmf(xs, rate * delta_t)

In [ ]:
bins = 20
outfile = None  # exploratory plot, not saved

plt.figure(figsize=(6, 4))

plt.hist(y_increments[0], bins=bins, rwidth=0.5, density=True, label='Data')
plt.plot(xs, ps, 'ko-', lw=1, label='Theory')
plt.ylabel('Frequency')
plt.xlabel('Increment')
plt.legend()

plt.tight_layout()
save_figure(outfile, outdir=outdir, dpi=dpi)
plt.show()

- **Remark**: from the plots above we cannot really tell _when_ the events happened in time. We only know that within a time interval $t$ there were $k$ events.

# 3. Interarrival times

Using the property of **interarrival times** of being iid exponentially distributed variables, we can now **simulate** also the times of _when_ events happen.

In [ ]:
seed = 10
rng = np.random.default_rng(seed)

In [ ]:
rate = 10
T = 2           # observation time window
n_events = 20   # number of interarrival times to draw

# generate interarrival times from an exponential distribution
taus = rng.exponential(1 / rate, n_events)

# compute successive sums to create a sequence of arrival times
ts = np.cumsum(taus)

# keep only events that happened within the observation window
ts = ts[ts < T]

event_idx = np.arange(len(ts))
len(ts), len(taus), len(event_idx)

In [ ]:
outfile = get_filename('pp_interarrival', lecture_id=lecture_id)

In [ ]:
plt.figure(figsize=(9, 5))

plt.subplot(2, 1, 1)
plt.scatter(ts, np.ones(len(ts)), s=100, edgecolors='black')
plt.ylabel('Event happening')
plt.xlabel('t')
plt.yticks([1])

plt.subplot(2, 1, 2)
plt.step(ts, event_idx)
plt.ylabel('N(t)')
plt.xlabel('t')

plt.tight_layout()
save_figure(outfile, outdir=outdir, dpi=dpi)
plt.show()

In [ ]:
def exponential_pdf(x, lmbda):
    return lmbda * np.exp(-lmbda * x)


xs = np.linspace(0, taus.max() + 5, 100)
exps = exponential_pdf(xs, rate)

Let's check that the empirical distribution matches the theoretical exponential distribution of the {$\tau_n$}

In [ ]:
bins = 20
outfile = get_filename('pp_interarrival_hist', lecture_id=lecture_id)

plt.figure(figsize=(6, 4))

plt.hist(taus, bins=bins, rwidth=0.5, density=True, label='Empirical')
plt.plot(xs, exps, label='Theoretical exponential')
plt.ylabel('Frequency')
plt.xlabel('Interarrival time')
plt.xlim([0, taus.max() * 1.1])
plt.legend()

plt.tight_layout()
save_figure(outfile, outdir=outdir, dpi=dpi)
plt.show()

### Can we simulate a PP in another way, besides using the {$\tau_n$}?
We need to simulate the times _when_ the events happen.
How?

Using the property of PP vs **uniform** distribution!

In [ ]:
seed = 10
rng = np.random.default_rng(seed)

rate, delta_t = 10, 1
T = 1000  # number of intervals to simulate

1. For each time interval $t$ we generate $n(t) = N(s+t) - N(s)$ events. 

2. Then, we generate $n(t)$ time indices {$t_k$} uniformly at random within $[s,s+t)$.

Example for one interval

In [ ]:
nt = rng.poisson(rate * delta_t)         # number of events within an interval delta_t
ts = np.sort(rng.uniform(0, delta_t, size=nt))
print(f"n(t) = {nt}\nt_k={ts}")

plt.figure(figsize=(6, 6))

plt.subplot(2, 1, 1)
plt.scatter(ts, np.ones(len(ts)), s=100, edgecolors='black')
plt.ylabel('Event happening')
plt.xlabel('t')

plt.subplot(2, 1, 2)
plt.step(ts, np.arange(1, nt + 1))
plt.plot(ts, nt * np.ones(len(ts)), ls='--', label=f"n(t)={nt}", c='black', lw=2)
plt.ylabel('N(t)')
plt.xlabel('t')
plt.legend()

plt.show()

Now sample for $T$ many intervals and collect the {$\tau_k$}

In [ ]:
taus = []  # interarrival times
for _ in range(T):
    nt = rng.poisson(rate * delta_t)  # number of events within an interval delta_t
    ts = np.sort(rng.uniform(0, delta_t, size=nt))
    taus.extend(np.diff(ts))
taus = np.array(taus)

Plot empirical distribution and check if it matches an exponential distribution of mean $1/\lambda$.

In [ ]:
xs = np.linspace(0, taus.max(), 100)
exps = exponential_pdf(xs, rate)

In [ ]:
plt.figure()
plt.hist(taus, bins=20, density=True, label='Empirical (histogram)')
plt.plot(xs, exps, 'k-', lw=2, label='Theoretical exponential (PDF)')
plt.xlim(0, taus.max())
plt.legend()
plt.show()

The histogram is sensitive to the choice of binning, which makes it hard to judge the fit.  
Let's try plotting with the cumulative distribution function (CDF) instead, which is bin-free.

In [ ]:
# compute the (experimental) CDF of the data
ecdf_x = np.sort(taus)
ecdf_y = np.arange(len(ecdf_x)) / float(len(ecdf_x))

# compute the (exact) CDF of the exponential
xs = np.linspace(0, taus.max(), 100)
exps = 1 - np.exp(-rate * xs)

fig, ax = plt.subplots(1, 1, figsize=(8, 5))
ax.plot(xs, exps, label='Theoretical (Exp) CDF')
ax.plot(ecdf_x, ecdf_y, label='Empirical CDF')
ax.set(xlabel=r'$\tau_k$ (interarrival time)', ylabel=r'$F(x)$')
ax.legend(fontsize=14)
plt.show()

## Q: what happens if you change the _uniform_ distribution to something else?
Let's try.  
We can for instance use a Gamma distribution.

In [ ]:
shape, scale = 1, 1  # mean = shape * scale, std = scale * sqrt(shape)
print(f"mean={shape * scale}, std={np.sqrt(shape) * scale}")

In [ ]:
nt = rng.poisson(rate * delta_t)  # number of events within an interval delta_t
ts = np.sort(rng.gamma(shape, scale, size=nt))
print(f"n(t) = {nt}, {len(ts)}")

plt.figure(figsize=(6, 6))

plt.subplot(2, 1, 1)
plt.scatter(ts, np.ones(len(ts)), s=100, edgecolors='black')
plt.ylabel('Event happening')
plt.xlabel('t')

plt.subplot(2, 1, 2)
plt.step(ts, np.arange(1, nt + 1))
plt.plot(ts, nt * np.ones(len(ts)), ls='--', label=f"n(t)={nt}", c='black', lw=2)
plt.ylabel('N(t)')
plt.xlabel('t')
plt.legend()

plt.show()

So far we do not notice much difference ...  
Let's generate more intervals.

In [ ]:
taus = []  # interarrival times
for _ in range(T):
    nt = rng.poisson(rate * delta_t)  # number of events within an interval delta_t
    ts = np.sort(rng.gamma(shape, scale, size=nt))
    taus.extend(np.diff(ts))
taus = np.array(taus)

Generate samples from a Gamma distribution to plot the theoretical CDF

In [ ]:
# compute the (experimental) CDF of the data
ecdf_x = np.sort(taus)
ecdf_y = np.arange(len(ecdf_x)) / float(len(ecdf_x))

# compute the (exact) CDF of the exponential, for comparison
xs = np.linspace(0, taus.max(), 100)
exps = 1 - np.exp(-rate * xs)

fig, ax = plt.subplots(1, 1, figsize=(8, 5))
ax.plot(xs, exps, label='Theoretical (Exp) CDF')
ax.plot(ecdf_x, ecdf_y, label='Empirical CDF')
ax.set(xlabel=r'$\tau_k$ (interarrival time)', ylabel=r'$F(x)$')
ax.legend(fontsize=14)
plt.show()

The two distributions don't match.  
That's because you need **uniformly** distributed times $t_k$ to get **exponentially** distributed interarrivals $\tau_k$.

# 4. Non-uniform Poisson Process
Let's simulate a NUPP and see how it looks like.

We consider as an example:
 - $\lambda(t) = \lambda_{max} \, \cos^2(2\pi\, t)$

This is a periodic behavior with $\lambda$ oscillating between 0 and its maximum value $\lambda_{max}$.

Simulating the process (no need to know why this is the way to simulate, but if you are interested you should look at "rejection sampling").

In [ ]:
T = 2
maxrate = 10


def rfun(t, maxrate: float = 10.):
    """Time-varying rate lambda(t) = maxrate * cos^2(2*pi*t)."""
    return maxrate * (np.cos(t * (2 * np.pi)) ** 2)


N_T = rng.poisson(maxrate * T)
ts = rng.uniform(0, T, size=N_T)

# thinning: keep each candidate point ts_i with probability rfun(ts_i) / maxrate
keep = rng.uniform(size=len(ts)) <= (rfun(ts, maxrate=maxrate) / maxrate)
ts_thin = ts[keep].copy()  # valid events for the NUPP

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(10, 3), sharex=True)

# draw rate
xs = np.linspace(0, T, 100)
axs[0].plot(xs, rfun(xs), 'b-')
axs[0].set_ylabel(r'$\lambda$')

# draw points: all candidates (grey) vs. accepted events after thinning (blue)
axs[1].scatter(ts, [0] * len(ts), c='k', s=50, alpha=0.1, label='Candidates')
axs[1].scatter(ts_thin, [0] * len(ts_thin), c='b', s=50, alpha=0.4, label='Accepted')
axs[1].set(xlim=[0, T])
axs[1].get_yaxis().set_visible(False)
axs[1].set_xlabel('t')
axs[1].legend(loc='upper right')

plt.tight_layout()
plt.show()